In [35]:

from datetime import datetime


class Account:
    """ Используем исключение, чтобы не создавать объект класса, когда не соблюдаются условия."""
    def __init__(self, account_holder, balance=0):
        if not isinstance(account_holder, str) or not account_holder.strip():
            raise ValueError("Имя владельца не может быть пустым")

        if isinstance(balance, bool) or not isinstance(balance, (int, float)):
            raise TypeError("Баланс должен быть числом")

        if balance < 0:
            raise ValueError("Начальный баланс не может быть отрицательным")

        self.holder = account_holder
        self._balance = float(balance)
        self.operations_history = []

    def _record_operation(self, operation_type, amount, status):
        #Записывает операцию в историю."""
        operation = {
            "type": operation_type,
            "amount": amount,
            "datetime": datetime.now(),
            "balance_after": self._balance,
            "status": status
        }

        self.operations_history.append(operation)

    @staticmethod
    # Статический метод
    def _validate_amount(amount):
        """ Используем исключение, чтобы не создавать объект класса, когда не соблюдаются условия"""
        if isinstance(amount, bool) or not isinstance(amount, (int, float)):
            raise TypeError("Сумма должна быть числом")

        if amount <= 0:
            raise ValueError("Сумма должна быть положительной")

    def deposit(self, amount):
        """Пополнение счёта."""
        self._validate_amount(amount)
        amount = float(amount)

        self._balance += amount
        self._record_operation("deposit", amount, "success")

        return self._balance

    def withdraw(self, amount):
        """Снятие средств с обычного счёта."""
        self._validate_amount(amount)
        amount = float(amount)

        if amount > self._balance:
            self._record_operation("withdraw", amount, "fail")
            return False

        self._balance -= amount
        self._record_operation("withdraw", amount, "success")

        return self._balance

    def get_balance(self):
        """Возвращает текущий баланс."""
        return self._balance

    def get_history(self):
        """Возвращает копию истории операций. Это исключает изменение истории и оптимизация скорости"""
        return [operation.copy()
                for operation in self.operations_history]


class CreditAccount(Account):
    def __init__(self, account_holder, balance=0, credit_limit=0):
        super().__init__(account_holder, balance)

        if isinstance(credit_limit, bool) or not isinstance(credit_limit, (int, float)):
            raise TypeError("Кредитный лимит должен быть числом")

        if credit_limit < 0:
            raise ValueError("Кредитный лимит не может быть отрицательным")

        self.credit_limit = float(credit_limit)

    def get_available_credit(self):
        """Возвращает доступную сумму с учетом кредитного лимита. Новый метод"""
        return max(0.0, self._balance + self.credit_limit)
        
    def withdraw(self, amount):
        """Снятие средств с учетом кредитного лимита."""
        """ Полная перегрузка метода """
        self._validate_amount(amount)
        amount = float(amount)

        if amount > self._balance + self.credit_limit:
            self._record_operation("withdraw", amount, "fail")
            self.operations_history[-1]["credit_used"] = (
                self._balance < 0
            )
            # Дополняем последнюю операцию информацией о сумме для погашения кредита
            self.operations_history[-1]["credit_amount"] = max(0.0, -self._balance)
            return False

        self._balance -= amount

        self._record_operation("withdraw", amount, "success")
        self.operations_history[-1]["credit_used"] = (
            self._balance < 0
        )
        self.operations_history[-1]["credit_amount"] = max(0.0, -self._balance)
        return self._balance

    def deposit(self, amount):
        """Пополнение кредитного счёта с учетом кредитных средств."""
         # Частичная перегрузка метода с использованнием  super() 
        self._balance = super().deposit(amount)
    
        # Проверяем, используются ли кредитные средства
        credit_used = self._balance < 0
    
        # Дополняем последнюю операцию информацией о кредите
        self.operations_history[-1]["credit_used"] = credit_used
        # Дополняем последнюю операцию информацией о сумме для погашения кредита
        self.operations_history[-1]["credit_amount"] = max(0.0, -self._balance)
    
        return self._balance

In [37]:
# Пример использования дебетового счёта
account = Account("Александр Пушкин", 1000)

account.deposit(500)
account.withdraw(200)
account.withdraw(2000)

print("Владелец:", account.holder)
print("Баланс:", account.get_balance())
print("История операций:")

for operation in account.get_history():
    print(operation)


print("\n--- Кредитный счёт ---")

credit_account = CreditAccount(
    "Иван Тургенев",
    balance=1000,
    credit_limit=5000
)

credit_account.withdraw(1000)
credit_account.withdraw(1000)
#credit_account.withdraw(4000)
#credit_account.withdraw(1000)

print("Владелец:", credit_account.holder)
print("Баланс:", credit_account.get_balance())
print("Доступный кредит:", credit_account.get_available_credit())

print("История кредитного счёта:")
for operation in credit_account.get_history():
    print(operation)

Владелец: Александр Пушкин
Баланс: 1300.0
История операций:
{'type': 'deposit', 'amount': 500.0, 'datetime': datetime.datetime(2026, 10, 26, 22, 29, 49, 954279), 'balance_after': 1500.0, 'status': 'success'}
{'type': 'withdraw', 'amount': 200.0, 'datetime': datetime.datetime(2026, 10, 26, 22, 29, 49, 954329), 'balance_after': 1300.0, 'status': 'success'}
{'type': 'withdraw', 'amount': 2000.0, 'datetime': datetime.datetime(2026, 10, 26, 22, 29, 49, 954371), 'balance_after': 1300.0, 'status': 'fail'}

--- Кредитный счёт ---
Владелец: Иван Тургенев
Баланс: -1000.0
Доступный кредит: 4000.0
История кредитного счёта:
{'type': 'withdraw', 'amount': 1000.0, 'datetime': datetime.datetime(2026, 10, 26, 22, 29, 49, 955028), 'balance_after': 0.0, 'status': 'success', 'credit_used': False, 'credit_amount': 0.0}
{'type': 'withdraw', 'amount': 1000.0, 'datetime': datetime.datetime(2026, 10, 26, 22, 29, 49, 955076), 'balance_after': -1000.0, 'status': 'success', 'credit_used': True, 'credit_amount': 1